# 🧠 Face Recognition using CNN
### Transfer Learning with VGG16
**Dataset:** Celebrity Face Image Dataset (Kaggle)

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install kaggle -q
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
import warnings; warnings.filterwarnings('ignore')
print("TensorFlow:", tf.__version__)

## 🔑 Step 2 — Upload kaggle.json API Key

In [ ]:
from google.colab import files
files.upload()   # upload your kaggle.json here

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle API key configured!")

## 📥 Step 3 — Download Dataset

In [ ]:
!kaggle datasets download -d vishesh1412/celebrity-face-image-dataset -p /content/
!unzip -q /content/celebrity-face-image-dataset.zip -d /content/data/
!ls /content/data/

## ⚙️ Step 4 — Configuration

In [ ]:
DATA_DIR = '/content/data/Celebrity Faces Dataset'
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)
print("Classes:", sorted(os.listdir(DATA_DIR)))

## 📁 Step 5 — Load Images

In [ ]:
images, labels = [], []
class_names = sorted(os.listdir(DATA_DIR))

for label in tqdm(class_names, desc='Loading'):
    folder = os.path.join(DATA_DIR, label)
    if not os.path.isdir(folder): continue
    for fname in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, fname))
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img); labels.append(label)

images, labels = np.array(images), np.array(labels)
print(f"Total images: {len(images)} | Classes: {len(class_names)}")

## 🔍 Step 6 — Visualize Samples

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for ax in axes.flat:
    idx = np.random.randint(len(images))
    ax.imshow(images[idx]); ax.set_title(labels[idx], fontsize=8); ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout(); plt.show()

## 🔄 Step 7 — Preprocess & Split

In [ ]:
X = images.astype('float32') / 255.0
le = LabelEncoder()
y_enc = le.fit_transform(labels)
y = to_categorical(y_enc, num_classes=len(class_names))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y_enc, random_state=SEED)
X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.1, random_state=SEED)
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 🏗️ Step 8 — Build Model (VGG16 + Custom Head)

In [ ]:
base = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
for layer in base.layers[:-4]: layer.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax')
])
model.compile(optimizer=optimizers.Adam(LEARNING_RATE),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 🚀 Step 9 — Train

In [ ]:
datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, zoom_range=0.1)

cb_list = [
    callbacks.EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    callbacks.ModelCheckpoint('/content/face_best.h5', save_best_only=True, verbose=1)
]
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train)//BATCH_SIZE,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, callbacks=cb_list)

## 📈 Step 10 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.suptitle('Face Recognition — Training History', fontsize=14)
plt.tight_layout(); plt.show()

## 🧪 Step 11 — Evaluate

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%  |  Test Loss: {loss:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Face Recognition', fontsize=14)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
model.save('/content/face_recognition_final.h5')
print("✅ Model saved to /content/face_recognition_final.h5")